# Convert 2025 Plot Census CSV to GeoJSON

Converts the 2025 census tree records to a GeoJSON point layer. Each row holds a
tree's **local** plot coordinates (`XCOORD`, `YCOORD`, in meters); these are anchored
to the plot's real center point (from `tree_plot_center_points.geojson`, reprojected
to UTM EPSG:32617) to get world positions. Only rows with a non-empty `NEWTAG` are
kept.

**Requirements:** Python 3.9+, `pandas`, `geopandas`.

**In:** `./download/current_survey/RP.Plot_census_data.2025.csv`,
`./download/tree_plot_center_points.geojson`
**Out:** `./download/current_survey/RP.Plot_census_data.2025.geojson` (EPSG:32617)

Trees are matched to plots on the **global plot number**: the census `PLOT` column
equals the geojson `id` (1..N, in the same order as the `Name` field increments).
Assumes the geojson point is the plot **center** and plots are ~100 m square
(`offset = 50` shifts from center to the corner the local coordinates are measured
from); adjust `offset` if the plot size differs.

In [7]:
import os
import pandas as pd
import geopandas as gpd

In [2]:
# Paths (relative, run from the project root).
census_path = "./download/current_survey/RP.Plot_census_data.2025.csv"
centers_path = "./download/tree_plot_center_points.geojson"
out_path = "./download/current_survey/RP.Plot_census_data.2025.geojson"

assert os.path.exists(census_path), f"Not found: {census_path}"
assert os.path.exists(centers_path), f"Not found: {centers_path}"

In [3]:
# Load plot center points and reproject to the survey UTM grid (EPSG:32617).
plots = gpd.read_file(centers_path)
if plots.crs is None:
    plots = plots.set_crs("EPSG:4326")   # centers are lat/lng if no CRS is embedded
plots = plots.to_crs("EPSG:32617")

# UTM center of each plot, as plain columns for the merge.
plots["plot_x"] = plots.geometry.x
plots["plot_y"] = plots.geometry.y

# The 'id' field is the global plot number (1..N), matching the census PLOT column.
plots["PLOT"] = plots["id"].astype(int)

In [4]:
# Load the census and keep only rows with a real NEWTAG (not NaN, not blank).
census = pd.read_csv(census_path)
need = {"UNIT", "PLOT", "XCOORD", "YCOORD", "NEWTAG"}
assert need <= set(census.columns), f"Census missing {need - set(census.columns)}; has {list(census.columns)}"

has_tag = census["NEWTAG"].notna() & census["NEWTAG"].astype(str).str.strip().ne("")
census = census[has_tag].copy()

In [5]:
# Anchor each tree to its plot center on the global PLOT number, then convert local
# coordinates to world UTM. The geometry is the plot center, so subtract offset
# (half a ~100 m plot) to align with the corner-referenced XCOORD/YCOORD.
census = census.merge(plots[["PLOT", "plot_x", "plot_y"]], on="PLOT", how="left")
assert census["plot_x"].notna().all(), "Some trees have no matching plot center."

offset = 50
census["world_x"] = census["XCOORD"] + census["plot_x"] - offset
census["world_y"] = census["YCOORD"] + census["plot_y"] - offset

In [6]:
# Build the point layer (UTM EPSG:32617), drop helper columns, write GeoJSON.
trees_gdf = gpd.GeoDataFrame(
    census,
    geometry=gpd.points_from_xy(census.world_x, census.world_y),
    crs="EPSG:32617",
).drop(columns=["plot_x", "plot_y", "world_x", "world_y"])

os.makedirs(os.path.dirname(out_path), exist_ok=True)
trees_gdf.to_file(out_path, driver="GeoJSON")
print(f"Wrote {len(trees_gdf)} tagged trees to {out_path}")

Wrote 685 tagged trees to ./download/current_survey/RP.Plot_census_data.2025.geojson
